# 05b · 선택적 정지 정리와 파산·배가 전략 (Optional Stopping, Gambler's Ruin and the Doubling Strategy)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e §5.3 (Gambling Strategies, Stopping Times), §5.4 (Applications: exit distributions/times) · Williams §10.6–10.10 ('you can't beat the system', stopped supermartingales, Doob's OST), §10.12 (hitting times for SRW) · Lawler 2e §5.3 (Optional Sampling) · G&S 4e §12.4–12.5 · Ross 12e §4.5.1 (Gambler's Ruin, 첫걸음 분석과 비교용) |
| 선수 노트북 | 05a (마팅게일 정의·동물원), 01c (흡수 사슬·파산 문제의 행렬 풀이), 00a (MC 도구) |
| 예상 소요 | 100분 |
| 상태 | draft |

05a에서 마팅게일은 "모든 **고정된** 시각 $n$에서 $E[M_n]=M_0$"인 과정이었다. 이 노트북의 질문은 하나다: **멈추는 시각을 내가 고르면 어떻게 되는가?** 답은 "공정한 게임은 어떤 멈춤 규칙으로도 공정하다 — 단, 무한한 시간이나 무한한 자본을 쓰지 않는 한"이다. 그 '단'을 정확히 적은 것이 선택적 정지 정리(Optional Stopping Theorem, OST)이고, 정리를 쓰면 01c에서 차분방정식으로 풀던 도박꾼의 파산 문제가 보존량 한 줄로 풀린다. 그리고 정리가 깨지는 두 반례(배가 전략, SRW의 $T_1$)에서 시뮬레이션이 어떻게 우리를 속이는지 본다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from scipy.special import comb
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.markov import (gamblers_ruin_matrix, gamblers_ruin_prob, gamblers_ruin_duration,
                          simulate_chain, hitting_times, random_walk_paths)

SEED, rng = rng_for("05b")
setup_plots()
N_PATHS = scale(20_000)      # 파산 경로 수 (FAST(SPKIT_FAST=1) 모드에서는 1/10 크기)
N_GAMES = scale(100_000)     # 배가 전략 게임 수
N_SWEEP = scale(4_000)       # 출발점 i를 바꿔 가며 쓰는 경로 수
N_RUIN_STEPS, N_RW_STEPS = 500, 1_000
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}, N_GAMES={N_GAMES}, N_SWEEP={N_SWEEP}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답해 보세요. 답은 접힌 블록에 있습니다.

**Q1.** (05a) $S_n^2-n$이 마팅게일인 이유를 한 줄로 계산하라.

<details><summary>정답</summary>

$E[(S_n+X_{n+1})^2\mid\mathcal F_n]=S_n^2+2S_n E[X_{n+1}]+E[X_{n+1}^2]=S_n^2+0+1$ 이므로 $E[S_{n+1}^2-(n+1)\mid\mathcal F_n]=S_n^2-n$.

</details>

**Q2.** (05a) 마팅게일이면 왜 모든 $n$에서 $E[M_n]=M_0$인가?

<details><summary>정답</summary>

탑 성질(tower property): $E[M_n]=E[E[M_n\mid\mathcal F_{n-1}]]=E[M_{n-1}]=\cdots=E[M_0]$. 이 노트북의 핵심 질문은 여기의 **고정된** $n$을 **확률적인** $T$로 바꿔도 되는가이다.

</details>

**Q3.** (05a) $e^{\theta S_n}$을 마팅게일로 만드는 정규화 상수와 그 이유는?

<details><summary>정답</summary>

$\cosh(\theta)^{-n}$. $E[e^{\theta X}]=\tfrac12(e^\theta+e^{-\theta})=\cosh\theta$이므로 $E[e^{\theta S_{n+1}}\mid\mathcal F_n]=e^{\theta S_n}\cosh\theta$. 이 노트북의 $(q/p)^{S_n}$도 같은 꼴이다: $e^{\theta}=q/p$로 두면 $E[(q/p)^X]=p\cdot\frac qp+q\cdot\frac pq=1$이라 정규화 상수가 1이다.

</details>

**Q4.** (04c) 상태 $x$의 평균 복귀시간이 40이면 장기 점유 비율은? 어떤 정리인가?

<details><summary>정답</summary>

$\pi(x)=1/E_x[T_x]=1/40=2.5\%$. 재생-보상 정리(복귀 사이클이 재생 사이클)로 증명한 관계. 복귀시각 $T_x$가 **정지시각**이라는 사실(강마르코프)이 그 증명의 출발점이었다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **정지시각**(stopping time): $T:\Omega\to\{0,1,2,\dots\}\cup\{\infty\}$이고 모든 $n$에 대해 $\{T\le n\}\in\mathcal F_n$. 말로는 "시각 $n$까지 본 것만으로 '이미 멈췄는지'를 판단할 수 있다". 이산시간에서는 $\{T=n\}\in\mathcal F_n$ (모든 $n$)과 동치.
  - 예: 첫 도달 시각 $T_A=\inf\{n:S_n\in A\}$; 상수 $c$; 정지시각 $\tau_1,\tau_2$의 $\tau_1\wedge\tau_2$, $\tau_1\vee\tau_2$; $T+k$ ($k\ge0$). (아래에서 $T_1$은 이런 일반 정지시각이 아니라 $+1$ 첫 도달 시각만을 뜻한다.)
  - 비-예: **마지막** 방문 시각, 경로의 **최댓값**이 되는 시각 — 그 시각이 지났는지 알려면 미래를 봐야 한다.
- **정지 과정**(stopped process): $M_{n\wedge T}$, $n\wedge T=\min(n,T)$. $T$ 이후에는 값이 얼어붙는다. 01c의 흡수 사슬 경로(`simulate_chain`이 만드는 것)가 바로 정지 과정이다 — 흡수되면 그 값에 머무른다.
- **도박꾼의 파산 설정**: $S_0=i$, 벽 $0$과 $N$, $T=\inf\{n:S_n\in\{0,N\}\}$, 상승확률 $p$, $q=1-p$, $r=q/p$. 이 노트북의 기본값 $N=10$, $i=3$, $p\in\{0.5,\,0.45\}$.
- **배가**(doubling, martingale betting) **전략**: 공정 동전에 1을 걸고, 질 때마다 판돈을 2배로, 처음 이기면 멈춘다. $k$번째 판에서 처음 이기면 순이익은 $2^{k-1}-(1+2+\cdots+2^{k-2})=2^{k-1}-(2^{k-1}-1)=1$. 첫 승리 시각 $T\sim\mathrm{Geom}(1/2)$, $P(T=k)=2^{-k}$, $E[T]=2$. **자본 한도** $K$판: $T>K$이면 $2^K-1$을 잃고 끝난다.
- **SRW의 $T_1=\inf\{n:S_n=1\}$**: $P(T_1<\infty)=1$ (01x, 재귀성)이지만 $E[T_1]=\infty$. 정확한 꼬리는 반사 원리로 $P(T_1>n)=P(S_n\in\{0,-1\})=\binom{n}{\lfloor n/2\rfloor}2^{-n}\sim\sqrt{2/(\pi n)}$ — 합이 발산하므로 $E[T_1]=\sum_{n\ge0}P(T_1>n)=\infty$.

### 2.2 정리 1 — 정지 마팅게일은 마팅게일이다 (Williams §10.9, Durrett 3e §5.3)

$M$이 마팅게일(서브/슈퍼)이고 $T$가 정지시각이면 $M_{n\wedge T}$도 마팅게일(서브/슈퍼)이다. 특히 **모든** $n$에서 $E[M_{n\wedge T}]=E[M_0]$.

가정이 왜 필요한가:
- **$T$가 정지시각**: 증명의 전부는 $1_{\{T>n\}}=1-1_{\{T\le n\}}$이 $\mathcal F_n$-가측이라는 한 줄이다. 미래를 보고 정하는 시각이면 '아는 것 꺼내기'를 쓸 수 없고 실제로 결론이 깨진다 — 최댓값 시각에 멈추면 $E[S_T]>0$ (05c E3에서 실험).
- **적분가능성**: $|M_{n\wedge T}|\le\max_{k\le n}|M_k|$이므로 유한 시각에서는 자동.

<details><summary>증명</summary>

먼저 항등식

$$M_{(n+1)\wedge T}-M_{n\wedge T}=1_{\{T>n\}}\,(M_{n+1}-M_n)$$

을 확인한다: $T\le n$이면 양변 모두 0 (좌변은 $M_T-M_T$), $T>n$이면 양변 모두 $M_{n+1}-M_n$. 이제 $1_{\{T>n\}}$은 $\mathcal F_n$-가측이므로 (아는 것 꺼내기)

$$E[M_{(n+1)\wedge T}-M_{n\wedge T}\mid\mathcal F_n]=1_{\{T>n\}}\,E[M_{n+1}-M_n\mid\mathcal F_n]=1_{\{T>n\}}\cdot0=0.$$

서브(슈퍼)마팅게일이면 마지막 항이 $\ge0$ ($\le0$)이고 $1_{\{T>n\}}\ge0$이므로 부호가 유지된다. 기댓값을 취해 $n$에 대해 더하면 $E[M_{n\wedge T}]=E[M_0]$. $\square$

이것이 Williams가 '시스템을 이길 수 없다'라고 부르는 것의 수학적 형태다: 베팅량 $H_n=1_{\{T>n\}}\in\{0,1\}$은 예측가능(previsible)한 전략이고, 예측가능한 유계 전략으로 마팅게일에 베팅한 누적 이득 $\sum_{k<n}H_k(M_{k+1}-M_k)$은 다시 마팅게일이다. 여기서는 $H_k=1_{\{T>k\}}$가 "$T$까지는 1씩 걸고 그 뒤로는 안 건다"는 전략이고, 그 누적 이득이 정확히 $M_{n\wedge T}-M_0$이다.

</details>

### 2.3 정리 2 — 선택적 정지 정리 (Optional Stopping Theorem, Doob; Williams §10.10, G&S 4e §12.5, Lawler 2e §5.3)

$M$ 마팅게일, $T$ 정지시각. 다음 중 **하나**가 성립하면 $E[M_T]=E[M_0]$:

- **(a)** $T\le c$ 유계;
- **(b)** $T<\infty$ a.s.이고 정지 과정이 유계: $|M_{n\wedge T}|\le c$ (모든 $n$);
- **(c)** $E[T]<\infty$이고 $\{T>n\}$ 위에서 증분이 유계: $|M_{n+1}-M_n|\le c$.

서브마팅게일이면 $E[M_T]\ge E[M_0]$, 슈퍼마팅게일이면 $\le$.

가정이 왜 필요한가:
- 정리 1만으로는 $E[M_{n\wedge T}]=M_0$ **까지만** 안다. 남은 문제는 $n\to\infty$에서 $E[M_{n\wedge T}]\to E[M_T]$가 되는가, 즉 **극한과 기댓값의 교환**이다. (a)는 극한이 필요 없는 경우이고, (b),(c)는 각각 유계수렴·지배수렴 정리의 조건이다.
- **(b)가 깨지는 예**: 배가 전략 — $M_{n\wedge T}$는 아래로 $-(2^n-1)$까지 내려가므로 유계가 아니다. SRW의 $T_1$ — $S_{n\wedge T_1}\le1$이지만 아래로 유계가 아니다.
- **(c)가 깨지는 예**: 배가 전략은 $E[T]=2<\infty$이지만 증분 $2^{n}$이 유계가 아니다. SRW의 $T_1$은 증분 $\pm1$이 유계이지만 $E[T_1]=\infty$. **(c)의 두 조건 중 하나만으로는 부족함**을 이 두 반례가 각각 보여준다.
- **$T<\infty$ a.s.**: 이것 없이는 $M_T$ 자체가 정의되지 않는다. (a),(c)에는 자동으로 포함되어 있다.

<details><summary>증명</summary>

세 경우 모두 좌변 $E[M_{n\wedge T}]=M_0$은 정리 1로 항상 참이다. 우변으로 넘어가는 방법만 다르다.

**(a)** $n=c$로 두면 $c\wedge T=T$이므로 $M_{c\wedge T}=M_T$. 극한이 필요 없다.

**(b)** $T<\infty$ a.s.이므로 $M_{n\wedge T}\to M_T$ a.s. (충분히 큰 $n$에서 $n\wedge T=T$). $|M_{n\wedge T}|\le c$이므로 **유계수렴정리**로 $E[M_{n\wedge T}]\to E[M_T]$. 좌변은 항상 $M_0$이므로 $E[M_T]=M_0$.

**(c)** 증분을 망원합으로 묶으면
$|M_{n\wedge T}-M_0|\le\sum_{k<n\wedge T}|M_{k+1}-M_k|\le c\,(n\wedge T)\le cT,$
그리고 $E[cT]<\infty$. 따라서 적분가능한 확률변수 $|M_0|+cT$가 $M_{n\wedge T}$를 지배하고, $E[T]<\infty$이면 $T<\infty$ a.s.이므로 $M_{n\wedge T}\to M_T$ a.s. **지배수렴정리**로 $E[M_{n\wedge T}]\to E[M_T]$.

서브/슈퍼의 경우 정리 1의 부등호가 그대로 극한을 통과한다. $\square$

</details>

### 2.4 정리 3 — 도박꾼의 파산, 마팅게일 풀이 (Durrett 3e §5.4; 01c의 행렬 풀이·Ross 12e §4.5.1의 첫걸음 풀이에 이은 세 번째 방법)

$S_0=i$, $T=\inf\{n:S_n\in\{0,N\}\}$.

- $p=\tfrac12$: $\;P_i(S_T=N)=\dfrac iN,\qquad E_i[T]=i(N-i).$
- $p\ne\tfrac12$, $r=q/p$: $\;P_i(S_T=N)=\dfrac{1-r^i}{1-r^N},\qquad E_i[T]=\dfrac{i}{q-p}-\dfrac{N}{q-p}\cdot\dfrac{1-r^i}{1-r^N}.$

가정이 왜 필요한가:
- **$E[T]<\infty$** (실제로는 기하 꼬리): 어느 상태에서든 연속 $N$번 상승하면 게임이 끝나므로 $P(T>kN)\le(1-p^N)^k$. 이것이 OST **(c)**를 $S_n^2-n$ (증분 $(S_n\pm1)^2-S_n^2-1=\pm2S_n$, 따라서 $|\text{증분}|\le 2N$)과 $S_n-(p-q)n$ (증분 $\le2$)에 적용하는 근거이다.
- **$S_n$과 $r^{S_n}$에는 (b)**: $0\le S_{n\wedge T}\le N$, $r^{S_{n\wedge T}}\le\max(1,r^N)$이므로 정지 과정이 유계다.
- **$p\ne\frac12$이면 $S_n$은 마팅게일이 아니다** (드리프트 $p-q$). 그래서 지수 마팅게일 $r^{S_n}$ ($E[r^{X}]=p\cdot\frac qp+q\cdot\frac pq=1$, Recall Q3)과 보정한 걸음 $S_n-(p-q)n$이 필요하다.

<details><summary>증명</summary>

**$p=\frac12$.** OST(b)를 $S_n$에: $i=E[S_0]=E[S_T]=N\cdot P_i(S_T=N)+0\cdot P_i(S_T=0)$ ⇒ $P_i(S_T=N)=i/N$.
OST(c)를 $S_n^2-n$에: $i^2=E[S_T^2-T]=N^2\cdot\dfrac iN-E[T]$ ⇒ $E_i[T]=Ni-i^2=i(N-i)$.

**$p\ne\frac12$.** OST(b)를 $r^{S_n}$에 ($P:=P_i(S_T=N)$): $r^i=E[r^{S_T}]=r^N P+r^0(1-P)$ ⇒ $P=\dfrac{1-r^i}{1-r^N}$.
OST(c)를 $S_n-(p-q)n$에: $i=E[S_T]-(p-q)E[T]=NP-(p-q)E[T]$ ⇒ $E_i[T]=\dfrac{NP-i}{p-q}=\dfrac{i-NP}{q-p}$, 정리하면 위의 식이다 (spkit의 `gamblers_ruin_duration`과 같은 식). $\square$

기본값 $N=10$, $i=3$: $p=\frac12$이면 $P=0.3$, $E[T]=21$; $p=0.45$이면 $r=11/9$, $P\approx0.1283$, $E[T]\approx17.17$ — 4절에서 코드로 계산하고 5절에서 비교한다. 01c의 풀이는 $(I-Q)^{-1}$을 구하는 것이었고, Ross 12e §4.5.1은 차분방정식 $P_i=pP_{i+1}+qP_{i-1}$을 푼다. 마팅게일 풀이는 **보존량 하나 = 방정식 하나**로, 차분방정식을 풀지 않는다.

</details>

### 2.5 정리 4 — 배가 전략의 역설과 자본 한도 (Williams §10.7, Durrett 3e §5.3)

배가 전략의 순손익 $M_n$은 마팅게일이고 $T\sim\mathrm{Geom}(1/2)$, $E[T]=2<\infty$이지만 $M_T=1$ a.s., 따라서 $E[M_T]=1\ne0=M_0$. 자본 한도 $K$판을 두면 $E[M_{T\wedge K}]=(1-2^{-K})\cdot1-2^{-K}(2^K-1)=0$. 사실 **모든 고정 $n$에서** $E[M_{n\wedge T}]=0$.

가정이 왜 필요한가:
- OST (a): $T$ 유계 아님. (b): $M_{n\wedge T}\ge-(2^n-1)$로 유계 아님. (c): 증분 $2^n$ 유계 아님. **세 조건이 모두 깨지므로** 결론이 깨져도 정리와 모순이 아니다.
- 자본 한도 $K$는 $T\wedge K$를 **유계 정지시각**으로 만들어 (a)를 회복시킨다 — 그래서 기대이익이 정확히 0이 된다. 카지노가 배가 전략을 두려워하지 않는 이유는 테이블 한도와 유한한 지갑이 (a)를 강제하기 때문이다.

<details><summary>증명</summary>

$n$판 후 아직 못 이겼으면($T>n$, 확률 $2^{-n}$) 손실은 $1+2+\cdots+2^{n-1}=2^n-1$; 이겼으면($T\le n$, 확률 $1-2^{-n}$) 이익 1. 따라서

$$E[M_{n\wedge T}]=(1-2^{-n})\cdot1-2^{-n}(2^n-1)=1-2^{-n}-1+2^{-n}=0.$$

이것은 정리 1의 결론과 일치한다. $n\to\infty$: $M_{n\wedge T}\to M_T=1$ a.s.이지만 기댓값은 0으로 고정 — 질량 $2^{-n}$이 $-2^n$으로 도망가서 극한과 기댓값이 교환되지 않는 전형적인 예다. Fatou의 보조정리 $E[\liminf X_n]\le\liminf E[X_n]$는 **아래로 유계**인 열에만 적용된다: $-M_{n\wedge T}\ge-1$에 적용하면 $E[\liminf(-M_{n\wedge T})]=-1<0=\liminf E[-M_{n\wedge T}]$로 등호 없이 성립하지만, $M_{n\wedge T}$ 자체는 아래로 무계이므로 $E[\lim M_{n\wedge T}]=1>0=\lim E[M_{n\wedge T}]$처럼 부등식의 방향이 반대로 깨진다. $\square$

</details>

### 2.6 직관

공정한 게임은 '멈추는 규칙'을 아무리 영리하게 짜도 공정하다 — 단, 무한한 시간이나 무한한 자본을 쓰지 않는 한. OST의 세 조건은 모두 "$M_{n\wedge T}\to M_T$에서 기댓값의 극한을 교환해도 되는가"를 보장하는 조건이며, 반례들은 예외 없이 **작은 확률로 거대한 값**이 생기는 구조를 가진다(배가 전략의 $-2^n$, $T_1$의 깊은 음수 경로).

파산 문제에서 마팅게일 풀이는 첫걸음 분석(01c)과 달리 차분방정식을 풀지 않는다: 보존량 하나 = 방정식 하나. $S_n$은 탈출 확률을, $S_n^2-n$은 기대 시간을, $r^{S_n}$은 편향 걸음의 탈출 확률을, $S_n-(p-q)n$은 편향 걸음의 기대 시간을 준다.

시뮬레이션 관점: 흡수 사슬의 경로는 정지 과정 그 자체이므로 시각 $n$에서의 열 평균은 항상 $M_0$이어야 한다. 그리고 바로 그것이 **절단**(censoring, 최대 스텝 $n_{\max}$에서 자르기)이 OST 위반을 **감추는** 이유이기도 하다: 코드는 언제나 $M_{T\wedge n_{\max}}$를 보고 있다.

### 2.7 함정 (traps)

1. **$T<\infty$ a.s.와 $E[T]<\infty$는 다르다.** $T_1$은 확률 1로 유한하지만 기댓값은 무한하고, OST가 실패한다($E[S_{T_1}]=1\ne0$).
2. **$E[M_{n\wedge T}]=M_0$은 항상 참이다**(정리 1). 실패할 수 있는 것은 오직 $n\to\infty$의 극한 교환이다.
3. **시뮬레이션의 절단 착시.** 최대 스텝 $n_{\max}$로 자르면 자동으로 $M_{T\wedge n_{\max}}$를 보는 것이고 그 평균은 $M_0$이다. 'OST가 항상 성립하는 것처럼' 보인다. 절단 비율과 절단된 경로의 값(예: $E[S_{1000}\mid T_1>1000]\approx-38.6$)을 반드시 함께 보고할 것.
4. **$r^{S_n}$의 밑은 $q/p$이다.** $p>\frac12$이면 밑이 1보다 작아 $[0,N]$ 위에서 1 이하, $p<\frac12$이면 밑이 1보다 커서 $r^N$까지 커진다 — 어느 쪽이든 $[0,N]$ 위에서 유계이므로 (b)가 적용된다.
5. **정지시각의 정의에는 필트레이션이 들어 있다.** '마지막으로 0을 방문한 시각'이나 '경로의 최댓값 시각'은 정지시각이 아니며, 그런 시각에서는 마팅게일 값의 평균이 $M_0$이 아니다.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요 — 특히 '왜 카지노는 배가 전략을 두려워하지 않는가'를 OST 조건 (b),(c)를 인용해 세 문장으로.)

## 3. Predict — 코드를 돌리기 전에 예측

코드를 돌리기 전에 아래 값을 먼저 적어 보세요. 손계산 힌트: 정리 3의 공식($r=11/9$), 정리 4의 $E[M_{n\wedge T}]$ 계산, $P(T_1>n)=\binom{n}{\lfloor n/2\rfloor}2^{-n}\approx\sqrt{2/(\pi n)}$. 마지막 항목을 적으면서 $E[T_1]$이 얼마인지도 함께 떠올려 보세요.

In [ ]:
predictions = {
    # N=10, i=3, p=1/2일 때 P_3(S_T = 10)은?
    "ruin_prob_p05": None,
    # 같은 설정에서 E_3[T]는?
    "ruin_time_p05": None,
    # p=0.45일 때 P_3(S_T = 10)은? (r = 11/9)
    "ruin_prob_p045": None,
    # p=0.45일 때 E_3[T]는? p=1/2보다 길까 짧을까?
    "ruin_time_p045": None,
    # 자본 한도 K=10판(최대 손실 1023)의 배가 전략에서 기대 순이익은?
    "doubling_capped_mean": None,
    # 대칭 SRW가 1000스텝 안에 +1에 도달할 확률 P(T_1 <= 1000)은?
    "p_T1_le_1000": None,
}

## 4. Simulate — 시뮬레이션

네 실험을 한다: 파산 $p=\frac12$ (4.1), 파산 $p=0.45$ (4.2), 배가 전략 (4.3), SRW의 $T_1$ (4.4). 앞의 둘은 OST가 **성립**하는 경우, 뒤의 둘은 **깨지는** 경우다. 먼저 spkit에 없는 작은 도우미를 정의한다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def stopped_mean_curve(paths):
    """각 열(시각 n)의 평균과 SE. 흡수 사슬 경로는 이미 정지 과정이므로 열 평균이 곧 E[M_{n∧T}]."""
    n = paths.shape[0]
    return paths.mean(0), paths.std(0, ddof=1) / np.sqrt(n)

def doubling_gain(T, cap=None):
    """첫 승리 시각 T(Geom(1/2))에서 배가 전략의 순이익. cap=K이면 K판까지만 허용."""
    if cap is None:
        return np.ones_like(T)
    return np.where(T <= cap, 1, -(2**cap - 1))

def P_T1_gt(n):
    """대칭 SRW: P(T_1 > n) = C(n, floor(n/2)) 2^{-n} (반사 원리)."""
    return comb(n, n // 2, exact=True) / 2**n

def E_min_T1_exact(n):
    """E[min(T_1, n)] = sum_{k<n} P(T_1 > k)."""
    return sum(P_T1_gt(k) for k in range(n))

### 4.1 파산 $p=\tfrac12$: 흡수 사슬의 경로 = 정지 과정 (Fig 1)

`gamblers_ruin_matrix(10, 0.5)`의 경로는 흡수되면 그 값에 머무르므로 **그 자체가 $S_{n\wedge T}$**이다. $T$는 `hitting_times`로 읽고, 500스텝 안에 모두 흡수되는지 확인한다($P(T>500)\approx\cos(\pi/10)^{500}\approx10^{-11}$이므로 절단은 사실상 없다). 네 가지를 추정한다: $P(S_T=10)$, $E[T]$, 그리고 정리 3의 두 보존량 $E[S_T]=3$, $E[S_T^2-T]=9$.

In [ ]:
N, i0 = 10, 3
P05 = gamblers_ruin_matrix(N, 0.5)
X05 = simulate_chain(P05, i0, N_RUIN_STEPS, rng, n_paths=N_PATHS)   # (N_PATHS, 501)
T = hitting_times(X05, {0, N})
assert np.all(np.isfinite(T)), "500스텝 안에 흡수되지 않은 경로가 있음"
S_T = X05[:, -1]                                                     # 흡수된 값 = S_T

est_hit05 = mc_proportion(S_T == N)
est_T05 = mc_estimate(T)
est_ST = mc_estimate(S_T)             # OST(b) on S_n  → 3
est_ST2 = mc_estimate(S_T**2 - T)     # OST(c) on S_n^2 - n → 9
print(f"P(S_T=10) = {est_hit05}   exact {gamblers_ruin_prob(i0, N, 0.5):.4f}")
print(f"E[T]      = {est_T05}   exact {gamblers_ruin_duration(i0, N, 0.5):.4f}, max T = {T.max():.0f}")
print(f"E[S_T]    = {est_ST}   exact {i0}")
print(f"E[S_T^2-T]= {est_ST2}   exact {i0**2}")

In [ ]:
m05, se05 = stopped_mean_curve(X05)
ns = np.arange(N_RUIN_STEPS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.hist(T, bins=np.arange(0, T.max() + 4, 4), density=True, alpha=0.7, label="MC")   # 모든 경로 포함
ax1.axvline(gamblers_ruin_duration(i0, N, 0.5), color="C3", ls="--", label="E[T] = i(N-i) = 21")
ax1.set(title="Histogram of ruin time T (N=10, i=3, p=1/2)", xlabel="T", ylabel="density", xlim=(0, 140))
ax1.legend()
ax2.plot(ns, m05, label="mean of S_{n AND T}")
ax2.fill_between(ns, m05 - 1.96 * se05, m05 + 1.96 * se05, alpha=0.25, label="95% band")
ax2.axhline(i0, color="C3", ls="--", label="S_0 = 3")
ax2.set(title="Stopped martingale: mean of S_{n AND T} vs n", xlabel="n", ylabel="mean")
ax2.legend()
plt.tight_layout()
del X05                                       # (N_PATHS, 501) 경로 배열은 더 쓰지 않으므로 해제

왼쪽: $T$의 분포는 21 근처에 몰려 있지 않고 오른쪽으로 길게 퍼져 있다(sd $\approx19.8$, 기하 꼬리; $T>140$인 경로는 밀도 계산에는 포함되지만 그림 밖에 있다). 오른쪽: 열 평균은 3에서 시작해 3에 머문다 — 정리 1의 $E[S_{n\wedge T}]=S_0$이 **모든** $n$에서 그대로 보인다. 큰 $n$에서는 거의 모든 경로가 흡수되어 $S_{n\wedge T}=S_T$이므로 이 곡선의 끝이 곧 $E[S_T]$다: OST(b)가 성립하는 경우다.

### 4.2 파산 $p=0.45$: 지수 마팅게일 $r^{S_n}$과 보정 걸음 (Fig 2)

$p\ne\frac12$이면 $S_n$은 마팅게일이 아니므로 $S_n$의 열 평균은 3에 머물지 않는다. 대신 $r^{S_n}$ ($r=11/9$)이 마팅게일이므로 $E[r^{S_{n\wedge T}}]=r^3\approx1.826$이 유지되어야 한다. $E[T]$는 `gamblers_ruin_duration`과 비교하는 동시에, 보정 걸음 $S_n-(p-q)n$의 OST에서 나오는 $(E[S_T]-i)/(p-q)$로도 계산해 같은 값이 나오는지 본다.

In [ ]:
p = 0.45
q, r = 1 - p, (1 - p) / p
P45 = gamblers_ruin_matrix(N, p)
X45 = simulate_chain(P45, i0, N_RUIN_STEPS, rng, n_paths=N_PATHS)
T45 = hitting_times(X45, {0, N})
assert np.all(np.isfinite(T45)), "500스텝 안에 흡수되지 않은 경로가 있음"
S_T45 = X45[:, -1]

est_hit45 = mc_proportion(S_T45 == N)
est_T45 = mc_estimate(T45)
est_r45 = mc_estimate(r**S_T45)                       # OST(b) on r^{S_n} → r^3
est_S45 = mc_estimate(S_T45)
ET_via_drift = (est_S45.mean - i0) / (p - q)           # OST(c) on S_n - (p-q)n: 같은 양, 다른 추정량
ET_via_drift_se = est_S45.se / abs(p - q)              # SE가 약 2배 크다 (1/|p-q| = 10 배율)
print(f"P(S_T=10)   = {est_hit45}   exact {gamblers_ruin_prob(i0, N, p):.6f}")
print(f"E[T]        = {est_T45}   exact {gamblers_ruin_duration(i0, N, p):.6f}")
print(f"E[T] via (E[S_T]-i)/(p-q) = {ET_via_drift:.4f} ± {ET_via_drift_se:.4f}  (같은 양의 더 시끄러운 추정량)")
print(f"E[r^S_T]    = {est_r45}   exact r^3 = {r**i0:.6f}")

In [ ]:
i_grid = np.arange(0, N + 1)
sweep = []                                   # i = 1..9 각각 N_SWEEP 경로 (메모리 절약: 루프)
for i in range(1, N):
    Xi = simulate_chain(P45, i, N_RUIN_STEPS, rng, n_paths=N_SWEEP)
    sweep.append(mc_proportion(Xi[:, -1] == N))
mr, ser = stopped_mean_curve(r**X45)         # (q/p)^{S_{n AND T}}: float 복사본 하나만 만든다
mS45 = X45.mean(0)                            # 같은 경로의 S_{n AND T} 열 평균 (복사 없음)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(i_grid, [gamblers_ruin_prob(i, N, p) for i in i_grid], color="C3", label="formula (1-r^i)/(1-r^N)")
ax1.errorbar(np.arange(1, N), [s.mean for s in sweep], yerr=[1.96 * s.se for s in sweep],
             fmt="o", capsize=3, label="MC (95% CI)")
ax1.set(title="P_i(hit 10) for p=0.45: MC vs formula", xlabel="start i", ylabel="probability")
ax1.legend()
ax2.plot(ns, mr, label="mean of (q/p)^{S_{n AND T}}")
ax2.fill_between(ns, mr - 1.96 * ser, mr + 1.96 * ser, alpha=0.25, label="95% band")
ax2.axhline(r**i0, color="C3", ls="--", label=f"(q/p)^3 = {r**i0:.3f}")
ax2.plot(ns, mS45 / i0 * r**i0, color="gray", lw=0.8, label="mean of S_{n AND T} (rescaled) - drifts")
ax2.set(title="E[(q/p)^{S_{n AND T}}] vs n stays at (q/p)^3", xlabel="n", ylabel="mean")
ax2.legend(fontsize=8)
plt.tight_layout()
del X45                                       # 이후 셀은 S_T45, T45만 쓴다

왼쪽: 출발점마다 독립 표본으로 추정한 $P_i(S_T=10)$이 공식 곡선 위에 놓인다(볼록한 모양: 음의 드리프트 때문에 10에 가까워져야 비로소 확률이 오른다). 오른쪽: $r^{S_{n\wedge T}}$의 평균은 $r^3$에 머무르지만, 회색선(같은 경로의 $S_{n\wedge T}$ 평균, 비교를 위해 $r^3$에 맞춰 스케일)은 아래로 흘러간다 — 마팅게일이 아닌 양은 보존되지 않는다.

### 4.3 배가 전략: 정리 4를 표본으로 (Fig 3)

첫 승리 시각 $T\sim\mathrm{Geom}(1/2)$를 `rng.geometric`으로 뽑는다. (a) 무한 자본이면 이익은 항상 1 — 대신 필요한 자본 $2^{T-1}-1$의 분포를 본다. (b) 한도 $K=10$: $T>10$이면 1023을 잃는다. (c) $E[M_{n\wedge T}]$를 $n=1,\dots,12$에서 추정해 정리 4의 "모든 $n$에서 0"을 확인하고 $E[M_T]=1$과 대비한다.

In [ ]:
K = 10
T_dbl = rng.geometric(0.5, size=N_GAMES)            # 첫 승리까지 판 수
gain_inf = doubling_gain(T_dbl)                     # 무한 자본: 항상 +1
capital = 2**(T_dbl - 1) - 1                        # 첫 승리 직전까지 잃은 돈 = 필요한 자본
gain_capped = doubling_gain(T_dbl, cap=K)
est_Tdbl = mc_estimate(T_dbl)
est_gain_capped = mc_estimate(gain_capped)
est_pwin = mc_proportion(T_dbl <= K)
n_grid = np.arange(1, 13)
M_n = [mc_estimate(doubling_gain(T_dbl, cap=n)) for n in n_grid]   # M_{n AND T}
est_M5 = M_n[4]
print(f"E[T] = {est_Tdbl}  (exact 2);  gain with infinite capital: always {gain_inf.min()}..{gain_inf.max()}")
print(f"max capital needed in {N_GAMES} games: {capital.max()}  (T_max = {T_dbl.max()})")
print(f"cap K=10: E[gain] = {est_gain_capped}  (exact 0),  P(win) = {est_pwin}  (exact {1 - 2.0**-K:.6f})")
print("E[M_{n AND T}] for n=1..12:", " ".join(f"{m.mean:+.2f}" for m in M_n))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
k_max = T_dbl.max()
edges = 2.0 ** np.arange(0, k_max + 1) - 0.5                 # 각 2^(T-1) 값을 한 칸씩
ax1.hist(capital + 1, bins=edges, weights=np.full(N_GAMES, 1 / N_GAMES), alpha=0.7,
         label="MC: P(T = k) per bin")
ax1.plot(2.0 ** np.arange(0, k_max), 0.5 ** np.arange(1, k_max + 1), "o", color="C3", ms=4,
         label="exact 2^{-k}")
ax1.set_xscale("log", base=2); ax1.set_yscale("log")
ax1.set(title="Capital needed before the first win, 2^(T-1)-1  (+1, log2 x)",
        xlabel="capital + 1 = 2^(T-1)", ylabel="probability P(T = k)")
ax1.legend()
ax2.errorbar(n_grid, [m.mean for m in M_n], yerr=[1.96 * m.se for m in M_n], fmt="o-", capsize=3,
             label="MC E[M_{n AND T}] (95% CI)")
ax2.axhline(0, color="C3", ls="--", label="E[M_{n AND T}] = 0 (Thm 1)")
ax2.axhline(1, color="k", ls=":", label="E[M_T] = 1")
ax2.set(title="Doubling: E[M_{n AND T}] vs n stays 0, E[M_T] = 1", xlabel="n (cap)", ylabel="mean gain")
ax2.legend(fontsize=8)
plt.tight_layout()

왼쪽: 필요 자본 $2^{k-1}-1$은 확률 $2^{-k}$ — 한 칸 오른쪽(자본 2배)으로 갈 때마다 확률은 절반이라 로그-로그 축에서 직선으로 줄어드는 기하 분포다. 필요 자본의 평균은 $\sum_k 2^{k-1}\cdot2^{-k}=\sum_k\frac12=\infty$: **무한 자본 전략의 기대 필요 자본은 무한**이다. 오른쪽: $E[M_{n\wedge T}]$는 모든 $n$에서 0 근처이지만 오차막대가 $n$과 함께 커진다(질량 $2^{-n}$의 $-2^n$이 분산 $\approx2^n$을 만든다). $n\to\infty$에서 점들은 0에 있는데 극한 $E[M_T]=1$: 극한과 기댓값이 교환되지 않는 모습이다.

### 4.4 SRW의 $T_1$: $T<\infty$ a.s.이지만 $E[T]=\infty$ (Fig 4)

`random_walk_paths`를 chunk 단위로 만들어 $T_1$ (1000스텝 안에 미도달이면 `inf`), $S_{10},S_{100},S_{1000}$, 그리고 정지 과정 $S_{n\wedge T_1}$의 열 합만 모은다(메모리 절약). $E[\min(T_1,n)]=\sum_{k<n}P(T_1>k)$는 정확히 계산되며 $n\to\infty$에서 $\approx2\sqrt{2n/\pi}\to\infty$. 절단된 2.5%의 경로가 어디에 있는지도 본다.

In [ ]:
n_chunks = 1 if fast() else 4
chunk = N_PATHS // n_chunks
T1_parts, S_parts, stopped_sum = [], [], np.zeros(N_RW_STEPS + 1)
ns_rw = np.arange(N_RW_STEPS + 1)
for _ in range(n_chunks):
    W = random_walk_paths(chunk, N_RW_STEPS, rng)
    t1 = hitting_times(W, 1)                                   # inf if never within 1000
    T1_parts.append(t1)
    S_parts.append(W[:, [10, 100, 1000]])
    stopped_sum += np.where(ns_rw[None, :] >= t1[:, None], 1, W).sum(0)   # S_{n AND T_1}
T1 = np.concatenate(T1_parts)
S_sel = np.concatenate(S_parts)                                # columns: S_10, S_100, S_1000
stopped_T1_mean = stopped_sum / (chunk * n_chunks)

est_pT1 = mc_proportion(np.isfinite(T1))
est_min = {n: mc_estimate(np.minimum(T1, n)) for n in (10, 100, 1000)}
censored = ~np.isfinite(T1)
print(f"P(T_1 <= 1000) = {est_pT1}   exact {1 - P_T1_gt(1000):.6f}")
for n in (10, 100, 1000):
    print(f"E[min(T_1,{n:4d})] = {est_min[n]}   exact {E_min_T1_exact(n):.5f}   2sqrt(2n/pi) = {2*np.sqrt(2*n/np.pi):.2f}")
print(f"E[S_{{1000 AND T_1}}] = {stopped_T1_mean[-1]:+.4f} (stays near 0);  censored fraction = {censored.mean():.4f}")
print(f"E[S_1000 | T_1 > 1000] = {S_sel[censored, 2].mean():.2f}   exact -P(T_1<=1000)/P(T_1>1000) = "
      f"{-(1 - P_T1_gt(1000)) / P_T1_gt(1000):.3f}")

In [ ]:
n_pts = np.array([1, 2, 3, 5, 10, 20, 50, 100, 200, 500, 1000])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.loglog(n_pts, [np.minimum(T1, n).mean() for n in n_pts], "o", label="MC E[min(T_1, n)]")
ax1.loglog(n_pts, [E_min_T1_exact(n) for n in n_pts], color="C3", label="exact sum_{k<n} P(T_1>k)")
ax1.loglog(n_pts, 2 * np.sqrt(2 * n_pts / np.pi), "k:", label="2 sqrt(2n/pi)")
ax1.set(title="E[min(T_1, n)] vs n (log-log): diverges", xlabel="n", ylabel="E[min(T_1, n)]")
ax1.legend()
ax2.plot(ns_rw, stopped_T1_mean, label="mean of S_{n AND T_1}")
ax2.axhline(0, color="C3", ls="--", label="S_0 = 0")
ax2.set(title="E[S_{n AND T_1}] stays 0 while P(T_1 <= n) -> 1", xlabel="n", ylabel="mean of S_{n AND T_1}")
ax2.set_ylim(-1.2, 1.2)
ax3 = ax2.twinx()
ax3.plot(ns_rw, [np.mean(T1 <= n) for n in ns_rw], color="gray", lw=0.8, label="P(T_1 <= n)")
ax3.set(ylabel="P(T_1 <= n)", ylim=(0, 1.05))
h1, l1 = ax2.get_legend_handles_labels(); h2, l2 = ax3.get_legend_handles_labels()
ax2.legend(h1 + h2, l1 + l2, loc="lower right", fontsize=8)
plt.tight_layout()

왼쪽: $E[\min(T_1,n)]$은 $\sqrt n$ 꼴로 끝없이 자란다 — $E[T_1]=\infty$의 시각화. 오른쪽: 97.5%의 경로가 이미 $+1$에서 멈춰 있는데도 열 평균은 0에 붙어 있다. 나머지 2.5%의 경로가 평균 $-38.6$의 깊은 음수에 있어 균형을 맞추기 때문이다. 이것이 함정 3의 절단 착시다: 절단 비율과 절단된 경로의 값을 같이 보지 않으면 "$E[S_{T_1}]=0$"이라고 잘못 읽는다.

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

닫힌 형식은 모두 코드로 계산한다: 정리 3은 `gamblers_ruin_prob/duration`과 $r^i$, 정리 4는 $(1-2^{-K})-2^{-K}(2^K-1)$, $T_1$은 반사 원리 합 `E_min_T1_exact`.

In [ ]:
exact = {
    "hit05": gamblers_ruin_prob(i0, N, 0.5), "T05": gamblers_ruin_duration(i0, N, 0.5),
    "ST": float(i0), "ST2": float(i0**2),
    "hit45": gamblers_ruin_prob(i0, N, p), "T45": gamblers_ruin_duration(i0, N, p), "r45": r**i0,
    "gain_capped": (1 - 2.0**-K) - 2.0**-K * (2**K - 1), "pwin": 1 - 2.0**-K, "Tdbl": 2.0,
    "M5": (1 - 2.0**-5) - 2.0**-5 * (2**5 - 1),
    "pT1": 1 - P_T1_gt(1000), "min100": E_min_T1_exact(100), "min1000": E_min_T1_exact(1000),
}
rows = [
    {"name": "P_3(S_T=10), p=1/2", "predicted": predictions["ruin_prob_p05"], "estimate": est_hit05, "exact": exact["hit05"]},
    {"name": "E_3[T], p=1/2", "predicted": predictions["ruin_time_p05"], "estimate": est_T05, "exact": exact["T05"]},
    {"name": "E[S_T], p=1/2 (OST on S_n)", "predicted": None, "estimate": est_ST, "exact": exact["ST"]},
    {"name": "E[S_T^2 - T], p=1/2 (OST on S_n^2-n)", "predicted": None, "estimate": est_ST2, "exact": exact["ST2"]},
    {"name": "P_3(S_T=10), p=0.45", "predicted": predictions["ruin_prob_p045"], "estimate": est_hit45, "exact": exact["hit45"]},
    {"name": "E_3[T], p=0.45", "predicted": predictions["ruin_time_p045"], "estimate": est_T45, "exact": exact["T45"]},
    {"name": "E[(q/p)^{S_T}], p=0.45", "predicted": None, "estimate": est_r45, "exact": exact["r45"]},
    {"name": "Doubling, cap K=10: E[gain]", "predicted": predictions["doubling_capped_mean"], "estimate": est_gain_capped, "exact": exact["gain_capped"]},
    {"name": "Doubling, cap K=10: P(win)", "predicted": None, "estimate": est_pwin, "exact": exact["pwin"]},
    {"name": "Doubling, no cap: E[T]", "predicted": None, "estimate": est_Tdbl, "exact": exact["Tdbl"]},
    {"name": "Doubling: E[M_{5 AND T}]", "predicted": None, "estimate": est_M5, "exact": exact["M5"]},
    {"name": "P(T_1 <= 1000)", "predicted": predictions["p_T1_le_1000"], "estimate": est_pT1, "exact": exact["pT1"]},
    {"name": "E[min(T_1, 100)]", "predicted": None, "estimate": est_min[100], "exact": exact["min100"]},
    {"name": "E[min(T_1, 1000)]", "predicted": None, "estimate": est_min[1000], "exact": exact["min1000"]},
]
Markdown(compare_table(rows))

각 행이 어긋난다면 무엇이 의심되는가:
- **P_3(S_T=10), p=1/2 / E[S_T]**: 두 행은 같은 정보($E[S_T]=10P$)다. 어긋나면 `hitting_times`의 목표 집합이나 `S_T = X[:, -1]`이 흡수 전 값을 읽고 있다(절단).
- **E_3[T], p=1/2 / E[S_T^2-T]**: 정리 3의 (c) 적용. 어긋나면 $T$가 흡수 시각이 아니라 '흡수 다음 시각'으로 세어졌다(off-by-one).
- **P_3(S_T=10), p=0.45 / E[(q/p)^{S_T}]**: 역시 같은 정보. `gamblers_ruin_matrix`의 $p$가 상승확률인지 하강확률인지 뒤바뀌면 $P\approx0.52$로 커진다 ($p=0.55$에서 $i=3$의 값).
- **E_3[T], p=0.45**: 4.2의 `ET_via_drift`와 `gamblers_ruin_duration`이 다르면 정리 3 증명의 부호 $(p-q)$가 틀린 것.
- **Doubling E[gain], P(win), E[T], E[M_{5∧T}]**: `doubling_gain`의 손실 $2^K-1$이 $2^K$로 되어 있거나 `geometric`이 "실패 횟수"(0부터)를 돌려주고 있다(numpy는 판 수, 1부터).
- **P(T_1 ≤ 1000), E[min(T_1,n)]**: `hitting_times(W, 1)`이 시각 0을 포함하거나 반사 원리 공식의 $\lfloor n/2\rfloor$가 틀렸다. $E[\min(T_1,1000)]$의 SE가 크다(sd $\approx176$) — 4 SE는 $\pm5$ 정도.

In [ ]:
assert_close(est_hit05, exact["hit05"], k=4, name="P(hit 10), p=1/2")
assert_close(est_T05, exact["T05"], k=4, name="E[T], p=1/2")
assert_close(est_ST, exact["ST"], k=4, name="E[S_T] = i")
assert_close(est_ST2, exact["ST2"], k=4, name="E[S_T^2 - T] = i^2")
assert_close(est_hit45, exact["hit45"], k=4, name="P(hit 10), p=0.45")
assert_close(est_T45, exact["T45"], k=4, name="E[T], p=0.45")
assert_close(est_r45, exact["r45"], k=4, name="E[(q/p)^{S_T}]")
assert_close(est_gain_capped, exact["gain_capped"], k=4, name="doubling capped E[gain]")
assert_close(est_pwin, exact["pwin"], k=4, name="doubling P(win)")
assert_close(est_Tdbl, exact["Tdbl"], k=4, name="doubling E[T]")
assert_close(est_M5, exact["M5"], k=4, name="E[M_{5 AND T}]")
assert_close(est_pT1, exact["pT1"], k=4, name="P(T_1 <= 1000)")
assert_close(est_min[100], exact["min100"], k=4, name="E[min(T_1,100)]")
assert_close(est_min[1000], exact["min1000"], k=4, name="E[min(T_1,1000)]")
print("all assert_close passed (k=4)")

## 6. 연습문제

### E1 계산

$N=8$, $i=2$에서 (a) $p=\frac12$, (b) $p=0.4$의 $P_2(S_T=8)$과 $E_2[T]$를 마팅게일로 구하라(어느 마팅게일에 OST의 어느 조건을 쓰는지 명시). (c) 왜 $p=0.4$에서 기대 지속시간이 더 짧은가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $S_n$에 OST(b): $P=i/N=2/8=0.25$. $S_n^2-n$에 OST(c): $E[T]=i(N-i)=2\cdot6=12$.

(b) $r=q/p=0.6/0.4=1.5$. $r^{S_n}$에 OST(b): $P=\dfrac{1-1.5^2}{1-1.5^8}=\dfrac{-1.25}{-24.6289}=0.050753$. $S_n-(p-q)n$에 OST(c): $E[T]=\dfrac{NP-i}{p-q}=\dfrac{8\cdot0.050753-2}{-0.2}=\dfrac{-1.593973}{-0.2}=7.969865$.

(c) 음의 드리프트 $p-q=-0.2$가 걸음을 0으로 빨리 밀어낸다. 두 벽 모두 흡수 경계이지만 가까운 쪽(0, 거리 2)으로 밀리므로 게임이 일찍 끝난다 — $E[T]$는 "어느 쪽이든 끝나는 시각"이지 "이기는 시각"이 아니다.

```python
for pp in (0.5, 0.4):
    print(pp, gamblers_ruin_prob(2, 8, pp), gamblers_ruin_duration(2, 8, pp))
# 0.5 0.25 12.0
# 0.4 0.050753370340999256 7.969865186360031
rr = 1.5; PP = (1 - rr**2) / (1 - rr**8); print(PP, (8 * PP - 2) / (0.4 - 0.6))   # 손계산 식 그대로
```

</details>

### E2 유도 후 시뮬레이션 검증

$N=10$, $i=5$, $p=0.55$ (이번엔 **양의** 드리프트). (a) $(9/11)^{S_n}$과 $S_n-0.1n$으로 $P_5(S_T=10)$, $E_5[T]$를 유도하라. $r^5$와 $1/(1+r^5)$의 관계도 확인하라. (b) `N_PATHS` 경로(500스텝)로 검증하고 `assert_close`를 둘 다 통과시켜라. (c) $p=0.45$, $i=5$의 $P_5(S_T=10)$은 얼마인가? 대칭성으로 답하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $r=9/11<1$. $r^{S_n}$에 OST(b): $P=\dfrac{1-r^5}{1-r^{10}}=\dfrac{1-r^5}{(1-r^5)(1+r^5)}=\dfrac1{1+r^5}=0.731717$ ($i=N/2$일 때만 생기는 인수분해). $S_n-(p-q)n=S_n-0.1n$에 OST(c): $E[T]=\dfrac{10P-5}{0.1}=23.171740$.

(b) 아래 코드. $P(T>500)$은 $p=\frac12$보다도 작으므로 절단 없음.

(c) $p\to1-p$, $i\to N-i$의 대칭으로 $P_5^{(0.45)}(S_T=10)=1-P_5^{(0.55)}(S_T=10)=0.268283$. 4.2의 스위프 그림에서 $i=5$ 점과 같다.

```python
X_e2 = simulate_chain(gamblers_ruin_matrix(10, 0.55), 5, 500, rng, n_paths=N_PATHS)
T_e2 = hitting_times(X_e2, {0, 10}); assert np.all(np.isfinite(T_e2))
est_P = mc_proportion(X_e2[:, -1] == 10); est_T = mc_estimate(T_e2)
r_e2 = 0.45 / 0.55
P_e2 = (1 - r_e2**5) / (1 - r_e2**10); ET_e2 = (10 * P_e2 - 5) / 0.1
print(P_e2, 1 / (1 + r_e2**5), ET_e2)                     # 0.731717 0.731717 23.171740
assert_close(est_P, P_e2, k=4, name="P_5(S_T=10), p=0.55")
assert_close(est_T, ET_e2, k=4, name="E_5[T], p=0.55")
print(1 - P_e2, gamblers_ruin_prob(5, 10, 0.45))           # 0.268283 (c)
```

</details>

### E3 가정을 깨보기

상한 벽 없이 SRW를 $+1$ 도달까지 달리게 하고($T_1$) "$E[S_{T_1}]$"을 시뮬레이션하라 — 4.4의 `T1`, `S_sel`을 재사용해도 좋다. (a) 왜 코드는 항상 0 근처를 주는가? 정확히 무엇을 추정하고 있는가? (b) 절단된 경로들만의 평균 $E[S_n\mid T_1>n]$을 $n=1000$에서 구하고, 정리 1로부터 그 값의 정확한 식을 유도하라. (c) $E[S_{T_1}]=1$과 정리 2를 대조해, (a)·(b)·(c) 중 어느 조건이 깨지는지 짚어라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) 코드는 $n_{\max}=1000$에서 자르므로 추정하는 것은 $E[S_{T_1}]$이 아니라 $E[S_{1000\wedge T_1}]$이고, 정리 1에 의해 이 값은 정확히 $S_0=0$이다. $T_1\le1000$인 97.5%의 경로는 값 1을, 나머지는 $S_{1000}$을 기여한다.

(b) 정리 1을 분해하면 $0=E[S_{n\wedge T_1}]=P(T_1\le n)\cdot1+E[S_n;\,T_1>n]$ ⇒
$$E[S_n\mid T_1>n]=-\frac{P(T_1\le n)}{P(T_1>n)},\qquad n=1000:\ -\frac{0.974775}{0.025225}=-38.643.$$
절단 비율 2.5%의 경로가 평균 $-38.6$의 큰 음수를 가져 전체 평균을 0으로 되돌린다.

(c) $E[S_{T_1}]=1\ne0$. OST (a): $T_1$ 유계 아님. (b): $S_{n\wedge T_1}\le1$이지만 아래로 무계. (c): 증분은 $\pm1$로 유계이지만 $E[T_1]=\infty$. 하나도 성립하지 않으므로 정리와 모순이 없다. 벽 $-N$을 추가하면(4.1의 파산 설정) (b)가 회복되고 $E[S_T]=0$이 **참**이 된다 — 그때는 $S_T=1$일 확률이 $N/(N+1)<1$이다.

```python
S1000 = S_sel[:, 2]
stopped_1000 = np.where(T1 <= 1000, 1, S1000)               # S_{1000 AND T_1}
print(mc_estimate(stopped_1000))                             # ≈ 0 ± 0.06 (정리 1)
cens = ~np.isfinite(T1)
print(cens.mean(), S1000[cens].mean())                       # ≈ 0.025, ≈ -38.6
print(-(1 - P_T1_gt(1000)) / P_T1_gt(1000))                  # -38.643
```

</details>

## 7. 정리

1. 정지시각은 '지금까지 본 것으로 결정되는 시각' $\{T\le n\}\in\mathcal F_n$; 정지 마팅게일 $M_{n\wedge T}$는 **항상** 마팅게일이고 $E[M_{n\wedge T}]=M_0$ (정리 1, 증명은 $1_{\{T>n\}}$이 $\mathcal F_n$-가측이라는 한 줄).
2. OST: $T$ 유계, 또는 정지 과정 유계 + $T<\infty$, 또는 $E[T]<\infty$ + 증분 유계이면 $E[M_T]=M_0$. 세 조건은 모두 $n\to\infty$에서 극한과 기댓값을 교환하는 조건(유계수렴·지배수렴)이다.
3. 파산 문제의 마팅게일 풀이: $S_n$ → $P=i/N$, $S_n^2-n$ → $E[T]=i(N-i)$, $(q/p)^{S_n}$과 $S_n-(p-q)n$ → 편향 공식. 차분방정식 없이 보존량 하나로 답 하나. 기본값 $(10,3)$: $0.3,\,21$ / $0.1283,\,17.17$.
4. 배가 전략($E[M_T]=1$)과 $T_1$($E[S_{T_1}]=1$)은 조건을 깨는 반례: 작은 확률의 거대한 손실이 기댓값을 $M_0$으로 되돌린다. 자본 한도 $K$를 두면 $T\wedge K$가 유계가 되어 정확히 공정해진다.
5. 시뮬레이션은 항상 절단된 $M_{T\wedge n_{\max}}$를 본다 — 절단 비율과 절단 경로의 값($E[S_{1000}\mid T_1>1000]=-38.6$)을 함께 보고해야 OST 위반을 놓치지 않는다.

**Trap 카드**
- Q: $T<\infty$ a.s.이고 마팅게일의 증분이 유계($|M_{n+1}-M_n|\le1$)이면 $E[M_T]=M_0$인가?
- A: 아니다. SRW의 $T_1$이 반례: 증분 $\pm1$, $T_1<\infty$ a.s., 그러나 $E[S_{T_1}]=1\ne0$. 조건 (c)는 $E[T]<\infty$를, (b)는 정지 과정 $M_{n\wedge T}$의 유계성을 요구하며 $T_1$은 둘 다 만족하지 않는다($E[T_1]=\infty$, 아래로 무계).

**다음 노트북: 05c** — OST(c)를 i.i.d. 합 $S_n-n\mu$에 적용하면 월드 항등식 $E[S_T]=E[X]\,E[T]$가 나온다(04a에서 재생 논법으로 봤던 것의 마팅게일 증명). 이를 거꾸로 써서 '패턴이 처음 나오는 시각'을 도박꾼 마팅게일로 계산한다(HTH $=10$ vs HHH $=14$).

log/progress.md 한 줄 템플릿:
`- [ ] 05b 선택적 정지 정리와 파산·배가 전략 — YYYY-MM-DD, 예측 6개 중 __개 적중, 막힌 곳: ____, 다시 볼 것: OST 세 조건과 두 반례(배가, T_1)의 대응표`